In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pengenalan kepada primitif

<Accordion>
<AccordionItem title="Versi pakej">

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## Mengapa Qiskit memperkenalkan primitif?
Sama seperti zaman awal komputer klasikal, apabila pembangun terpaksa memanipulasi daftar CPU secara terus, antara muka awal kepada QPU hanya mengembalikan data mentah dari elektronik kawalan.
Ini tidak menjadi isu besar apabila QPU berada dalam makmal dan hanya membenarkan akses langsung oleh penyelidik.
Menyedari bahawa kebanyakan pembangun tidak akan dan tidak seharusnya biasa dengan menyuling data mentah tersebut kepada 0 dan 1, Qiskit memperkenalkan `backend.run`, abstraksi pertama untuk mengakses QPU di awan. Ini membolehkan pembangun beroperasi pada format data yang biasa dan menumpukan pada gambaran yang lebih besar.

Apabila akses kepada QPU menjadi lebih meluas, dan dengan lebih banyak algoritma kuantum dibangunkan, keperluan untuk abstraksi peringkat lebih tinggi timbul semula. Sebagai respons, Qiskit memperkenalkan antara muka primitif, yang dioptimumkan untuk dua tugas teras dalam pembangunan algoritma kuantum: anggaran nilai jangkaan (`Estimator`) dan pensampelan Circuit (`Sampler`). Matlamatnya sekali lagi adalah untuk membantu pembangun lebih menumpukan pada inovasi dan kurang pada penukaran data. Antara muka primitif menggantikan antara muka `backend.run`, kerana `Sampler` menyediakan akses perkakasan langsung yang sama seperti yang ditawarkan oleh `backend.run`.

## Apakah primitif?
Sistem pengkomputeran dibina di atas pelbagai lapisan abstraksi. Abstraksi membolehkan anda menumpukan pada tahap perincian tertentu yang relevan dengan tugas yang sedang dijalankan. Semakin hampir anda ke perkakasan, semakin rendah tahap abstraksi yang anda perlukan (contohnya, anda mungkin perlu memindahkan atau memanipulasi data pada peringkat arahan CPU). Semakin kompleks tugas yang ingin anda lakukan, semakin tinggi tahap abstraksinya (contohnya, anda mungkin menggunakan pustaka pengaturcaraan untuk melakukan pengiraan algebra).

Dalam konteks ini, *primitif* ialah arahan pemprosesan terkecil, blok binaan paling mudah yang boleh digunakan untuk mencipta sesuatu yang berguna untuk tahap abstraksi tertentu.

Kemajuan terkini dalam pengkomputeran kuantum telah meningkatkan keperluan untuk bekerja pada tahap abstraksi yang lebih tinggi. Apabila bidang ini bergerak ke arah unit pemprosesan kuantum (QPU) yang lebih besar dan aliran kerja yang lebih kompleks, tumpuan beralih daripada berinteraksi dengan isyarat qubit individu kepada melihat peranti kuantum sebagai sistem yang menjalankan tugas yang diperlukan.

Dua tugas yang paling biasa untuk komputer kuantum adalah pensampelan keadaan kuantum dan pengiraan nilai jangkaan. Tugas-tugas ini mendorong reka bentuk primitif Qiskit: **Estimator** dan **Sampler**.

- Estimator mengira nilai jangkaan observable berkenaan keadaan yang disediakan oleh Circuit kuantum.
- Sampler mengambil sampel daftar output daripada pelaksanaan Circuit kuantum.

Ringkasnya, model pengkomputeran yang diperkenalkan oleh primitif Qiskit membawa pengaturcaraan kuantum selangkah lebih hampir ke tahap pengaturcaraan klasikal hari ini, di mana tumpuan kurang pada perincian perkakasan dan lebih pada keputusan yang ingin dicapai.

## Definisi dan pelaksanaan primitif
Terdapat dua jenis primitif Qiskit: kelas asas, dan pelaksanaannya. Primitif Estimator dan Sampler ditakrifkan oleh kelas asas primitif sumber terbuka yang terdapat dalam Qiskit SDK (dalam modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)). Pembekal (seperti Qiskit Runtime) boleh menggunakan kelas asas ini untuk memperoleh pelaksanaan Sampler dan Estimator mereka sendiri. Kebanyakan pengguna akan berinteraksi dengan pelaksanaan pembekal, bukan primitif asas.

### Kelas asas
Primitif `Base` ialah kelas abstrak yang menakrifkan antara muka biasa untuk melaksanakan primitif. Semua kelas lain dalam modul [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) mewarisi daripada kelas asas ini. Pembangun harus menggunakan ini jika mereka berminat untuk mencipta model pelaksanaan berasaskan primitif mereka sendiri untuk pembekal tertentu. Kelas-kelas ini mungkin juga berguna bagi mereka yang mahu melakukan pemprosesan yang sangat disesuaikan dan mendapati pelaksanaan primitif sedia ada terlalu mudah untuk keperluan mereka. Pengguna umum tidak akan menggunakan kelas asas secara langsung.

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) dan [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - Walaupun primitif V1 masih boleh digunakan, panduan ini menumpukan pada primitif V2 kerana ia adalah yang terkini dan lebih kerap digunakan.

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) dan [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Primitif rujukan Qiskit mengikuti spesifikasi antara muka ini.

<span id="implementations"></span>
### Pelaksanaan
Semua primitif dicipta daripada kelas asas; oleh itu, mereka mempunyai struktur dan penggunaan am yang sama. Contohnya, format input untuk semua primitif Estimator adalah sama. Walau bagaimanapun, terdapat perbezaan dalam pelaksanaan yang menjadikan setiap satu unik.

Berikut adalah pelaksanaan kelas asas primitif:

- [Primitif Qiskit Runtime](/guides/qiskit-runtime-primitives), [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) dan [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2), menyediakan pelaksanaan yang lebih canggih (contohnya, dengan memasukkan mitigasi ralat) sebagai perkhidmatan berasaskan awan. Pelaksanaan primitif asas ini digunakan untuk mengakses perkakasan IBM Quantum&reg;.

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) dan [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Pelaksanaan rujukan primitif yang menggunakan simulator terbina dalam Qiskit. Ia dibina dengan modul Qiskit [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information), menghasilkan keputusan berdasarkan simulasi statevector yang ideal. Ia diakses melalui Qiskit. Lihat [Simulasi tepat dengan primitif Qiskit](/guides/simulate-with-qiskit-sdk-primitives) untuk butiran penggunaan.

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) dan [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - Anda boleh menggunakan kelas ini untuk "membungkus" mana-mana sumber pengkomputeran kuantum menjadi primitif. Ini membolehkan anda menulis kod bergaya primitif untuk pembekal yang belum lagi mempunyai antara muka berasaskan primitif. Kelas-kelas ini boleh digunakan sama seperti Sampler dan Estimator biasa, kecuali ia perlu dimulakan dengan argumen `backend` tambahan untuk memilih komputer kuantum yang hendak digunakan. Ia diakses menggunakan Qiskit. Lihat panduan [primitif backend](/guides/get-started-with-backend-primitives) untuk maklumat lanjut.

## Pilihan
Anda boleh menghantar pilihan kepada primitif untuk menyesuaikannya dengan keperluan anda. Walaupun antara muka kaedah `run()` primitif adalah sama merentasi semua pelaksanaan, pilihan mereka tidak. Rujuk rujukan API untuk pelaksanaan primitif tertentu untuk mengetahui pilihan yang disokongnya.

Contohnya, rujuk topik [Pilihan Estimator](/guides/estimator-options) dan [Pilihan Sampler](/guides/sampler-options) untuk mengetahui pilihan untuk primitif Qiskit Runtime, atau lihat [rujukan API Qiskit Aer](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) untuk pilihan primitif Qiskit Aer.

## Manfaat primitif Qiskit
Dengan primitif, pengguna Qiskit boleh menulis kod kuantum untuk QPU tertentu tanpa perlu mengurus setiap perincian secara eksplisit. Selain itu, disebabkan lapisan abstraksi tambahan, anda mungkin dapat mengakses keupayaan perkakasan lanjutan pembekal tertentu dengan lebih mudah. Contohnya, dengan primitif Qiskit Runtime, anda boleh memanfaatkan kemajuan terkini dalam mitigasi dan penindasan ralat dengan menukar pilihan seperti [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) primitif, daripada membina pelaksanaan teknik ini sendiri.

Bagi pembekal perkakasan, melaksanakan primitif secara asli bermakna anda boleh menyediakan pengguna anda dengan cara "sedia guna" yang lebih mudah untuk mengakses ciri perkakasan anda seperti teknik pasca-pemprosesan lanjutan. Oleh itu, pengguna anda lebih mudah mendapat manfaat daripada keupayaan terbaik perkakasan anda.

## Langkah seterusnya
> **Tip:** - Fahami [input dan output primitif](/guides/primitive-input-output).
> - Semak [contoh](/guides/simulate-with-qiskit-sdk-primitives) yang terperinci.
> - Berlatih dengan primitif melalui [pelajaran Fungsi kos](/learning/courses/variational-algorithm-design/cost-functions) dalam IBM Quantum Learning.
> - Semak [Cipta pembekal](/guides/create-a-provider) untuk mempelajari cara melaksanakan primitif Sampler dan Estimator anda sendiri.
> - Lihat [rujukan API](https://docs.quantum.ibm.com/api/qiskit/primitives).
> - Baca [Migrasi ke primitif V2](/guides/v2-primitives).
> - Ketahui tentang [primitif Qiskit Runtime](/guides/qiskit-runtime-primitives), yang digunakan untuk menjalankan Circuit pada QPU IBM.